### Snowflake JMX

- generate a public private key and set it up in your user. 

`openssl genrsa 2048 | openssl pkcs8 -topk8 -v2 des3 -inform PEM -out rsa_key.p8`

`openssl rsa -in rsa_key.p8 -pubout -out rsa_key.pub`

`ALTER USER svc_jmeter SET RSA_PUBLIC_KEY='MIIBIjANBgkq…';`

- add it to the parameters above


In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Create Widgets (all text) ---

dbutils.widgets.removeAll()

dbutils.widgets.text("sf_account_locator", "", "Snowflake Account Locator")
dbutils.widgets.text("sf_region", "", "Snowflake Region")
dbutils.widgets.text("sf_warehouse", "", "Snowflake Warehouse Size (XS, S, M, L, ...)")
dbutils.widgets.text("scale", "100", "Scale Factor (GB)")
dbutils.widgets.text("sf_database", "", "Snowflake Database")
dbutils.widgets.text("sf_schema", "PUBLIC", "Snowflake Schema")
dbutils.widgets.text("sf_role", "PUBLIC", "Snowflake Role")
dbutils.widgets.text("sf_user", "", "Snowflake User")
dbutils.widgets.text("sf_auth_key", "", "Snowflake Auth Key")
dbutils.widgets.text("table_prefix", "", "Table Prefix")
dbutils.widgets.text("sf_strategy", "native_gen1", "Snowflake Strategy (e.g. native_gen1)")
dbutils.widgets.text("sf_strategy_slug", "", "Snowflake Strategy Slug (auto-computed if blank)")
dbutils.widgets.text("dbx_catalog", "tpcds_sf_catalog", "Databricks Catalog (for volume)")
dbutils.widgets.text("dbx_schema", "tpcds", "Databricks Schema (for volume)")
dbutils.widgets.text("dbx_volume", "tpcds_raw_data", "Volume name")

In [ ]:
import os
# --- Retrieve Values, Set Env Vars, and Build Dictionary ---

params = dbutils.widgets.getAll()

for k,v in params.items():
    os.environ[f"W_{k.upper()}"] = v

# --- Validation Check (all required) ---

if params["sf_account_locator"] == "":
    raise RuntimeError("Validation Error: Parameter 'sf_account_locator' is required but is empty.")
if params["sf_region"] == "":
    raise RuntimeError("Validation Error: Parameter 'sf_region' is required but is empty.")
if params["sf_warehouse"] == "":
    raise RuntimeError("Validation Error: Parameter 'sf_warehouse' is required but is empty.")
if params["sf_database"] == "":
    raise RuntimeError("Validation Error: Parameter 'sf_database' is required but is empty.")
if params["sf_schema"] == "":
    raise RuntimeError("Validation Error: Parameter 'sf_schema' is required but is empty.")
if params["sf_role"] == "":
    raise RuntimeError("Validation Error: Parameter 'sf_role' is required but is empty.")
if params["sf_user"] == "":
    raise RuntimeError("Validation Error: Parameter 'sf_user' is required but is empty.")
if params["sf_auth_key"] == "":
    raise RuntimeError("Validation Error: Parameter 'sf_auth_key' is required but is empty.")
if params["scale"] == "":
    raise RuntimeError("Validation Error: Parameter 'scale' is required but is empty.")


In [ ]:
import os

# Write the private key and resolve to absolute path for JDBC driver
volume_base = f"/Volumes/{params['sf_database']}/{params['sf_schema']}/tpcds_raw_data".replace("TPCDS/BENCHMARK", "tpcds_sf_catalog/tpcds") if False else f"/Volumes/{params.get('dbx_catalog', 'tpcds_sf_catalog')}/{params.get('dbx_schema', 'tpcds')}/{params.get('dbx_volume', 'tpcds_raw_data')}"
os.makedirs(f"{volume_base}/test_files", exist_ok=True)
auth_key_path = os.path.join(volume_base, "test_files", "rsa_key.p8")
with open(auth_key_path, "w") as f:
    f.write(params["sf_auth_key"])

In [ ]:
import os

current_path = os.getcwd()
table_prefix = params.get("table_prefix", "").rstrip("_")
conf_dir = f"{volume_base}/test_files/{table_prefix}" if table_prefix else f"{volume_base}/test_files"
os.makedirs(conf_dir, exist_ok=True)

# Build the Snowflake JDBC hostname.
acct = params["sf_account_locator"]
region = params["sf_region"]
if "-" in acct:
    sf_jdbc_host = f"{acct}.snowflakecomputing.com"
else:
    sf_jdbc_host = f"{acct}.{region}.snowflakecomputing.com"

# Resolve warehouse name to match what 5.1a/b/c create via
# find_or_create_snowflake_warehouse: TPCDS_{scale}_{size}_{strategy_slug}
size_token = params["sf_warehouse"].strip().upper()
slug_token = (params.get("sf_strategy_slug") or "").strip().upper() or strategy_slug(params.get("sf_strategy",""))
sf_warehouse = f"TPCDS_{params['scale']}_{size_token}_{slug_token}"
print(f"Resolved warehouse name: {sf_warehouse}")

# RSA key for JDBC auth (staged at runtime by 5.1a/b/c via shutil.copy2 -> /tmp/tpcds/test_conf/)
auth_key_path = "/tmp/tpcds/test_conf/rsa_key.p8"

fill_params = {
    "JDBC_DRIVER": "/tpcds/sf_driver.jar",
    "SF_JDBC_HOST": sf_jdbc_host,
    "SF_WAREHOUSE": sf_warehouse,
    "SF_DATABASE": params["sf_database"],
    "SF_SCHEMA": params["sf_schema"],
    "SF_ROLE": params["sf_role"],
    "SF_USER": params["sf_user"],
    "SF_AUTH_KEY": auth_key_path,
    "CURPATH": "/tmp/tpcds"  # runtime dir where 5a/5b/5c stage test_conf/ and JMeter writes results/,
}

# --- Ingestion JMX ---
template = get_jmx_template("SNOWFLAKE", "INGESTION")
jmx = fill_template(template, fill_params)
with open(f"{conf_dir}/snowflake_ingestion.jmx", "w") as f:
    f.write(jmx)

# --- Power Test JMX ---
template = get_jmx_template("SNOWFLAKE", "POWER_TEST")
jmx = fill_template(template, fill_params)
with open(f"{conf_dir}/snowflake_power_test.jmx", "w") as f:
    f.write(jmx)

# --- Concurrency JMX ---
template = get_jmx_template("SNOWFLAKE", "CONCURRENCY")
jmx = fill_template(template, fill_params)
with open(f"{conf_dir}/snowflake_concurrency.jmx", "w") as f:
    f.write(jmx)

print(f"Generated 3 Snowflake JMX files in {conf_dir}")

In [ ]:
import os

current_path = os.getcwd()

generate_concurrency_streams(
    perf_queries_tsv=f"{conf_dir}/tpcds-queries-perf-snowflake.tsv",
    concurrency_json=f"{current_path}/resources/tpcds-concurrency-query-order.json",
    output_dir=f"{conf_dir}/tpcds_snowflake_concurrency_streams",
)